In [ ]:
#Task 1
# Variables: The robots position at each step
# Domains: xi, yi in range 0-4 (since the grid is 5x5)
# Constraints: Start at (1,1) and end at (4,4), only move to adjacent cells, 
# cannot move through walls (0s in the grid), and cannot revisit any cell.
from ortools.sat.python import cp_model
import math

grid = [
    [1, 1, 1, 1, 1],
    [1, 1, 1, 0, 1],
    [1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [1, 1, 1, 1, 1]
]

size = 5
flat_grid = [v for row in grid for v in row]
start = (0, 0)
target = (3, 3)
max_steps = 12

best_path = None
best_cost = None

for N in range(1, max_steps + 1):
    model = cp_model.CpModel()

    gx = [model.NewIntVar(0, size - 1, f"x_{i}") for i in range(N + 1)]
    gy = [model.NewIntVar(0, size - 1, f"y_{i}") for i in range(N + 1)]
    pos = [model.NewIntVar(0, size * size - 1, f"pos_{i}") for i in range(N + 1)]

    model.Add(gx[0] == start[0])
    model.Add(gy[0] == start[1])
    model.Add(gx[N] == target[0])
    model.Add(gy[N] == target[1])

    for i in range(N + 1):
        model.Add(pos[i] == gx[i] * size + gy[i])
        model.AddElement(pos[i], flat_grid, 1)

    for i in range(N):
        dx = model.NewIntVar(-1, 1, f"dx_{i}")
        dy = model.NewIntVar(-1, 1, f"dy_{i}")
        adx = model.NewIntVar(1, 1, f"abs_dx_{i}")
        ady = model.NewIntVar(1, 1, f"abs_dy_{i}")
        model.Add(dx == gx[i + 1] - gx[i])
        model.Add(dy == gy[i + 1] - gy[i])
        model.AddAbsEquality(adx, dx)
        model.AddAbsEquality(ady, dy)

    model.AddAllDifferent(pos)

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        best_path = [(solver.Value(gx[i]) + 1, solver.Value(gy[i]) + 1) for i in range(N + 1)]
        best_cost = N * math.sqrt(2)
        break

if best_path:
    print("Shortest Diagonal Path:", best_path)
    print("Total Cost:", best_cost)
else:
    print("No valid path found.")

Shortest Diagonal Path: [(1, 1), (2, 2), (3, 3), (4, 4)]
Total Cost: 4.242640687119286


In [ ]:
#Task 2
from ortools.sat.python import cp_model
from collections import deque

island = [
    [1, 1, 0, 0, 0],
    [1, 1, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

rows = len(island)
cols = len(island[0])

model = cp_model.CpModel()
land = [[model.NewBoolVar(f"land_{i}_{j}") for j in range(cols)] for i in range(rows)]

for i in range(rows):
    for j in range(cols):
        model.Add(land[i][j] == island[i][j])

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    grid = [[solver.Value(land[i][j]) for j in range(cols)] for i in range(rows)]
    visited = [[False for _ in range(cols)] for _ in range(rows)]
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    best_area = 0
    best_perimeter = 0

    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1 and not visited[i][j]:
                q = deque([(i, j)])
                visited[i][j] = True
                area = 0
                perimeter = 0

                while q:
                    x, y = q.popleft()
                    area += 1

                    for dx, dy in dirs:
                        nx, ny = x + dx, y + dy
                        if nx < 0 or nx >= rows or ny < 0 or ny >= cols or grid[nx][ny] == 0:
                            perimeter += 1
                        elif not visited[nx][ny]:
                            visited[nx][ny] = True
                            q.append((nx, ny))

                if area > best_area:
                    best_area = area
                    best_perimeter = perimeter

    print("Largest Landmass Area:", best_area)
    print("Perimeter:", best_perimeter)
else:
    print("No solution found.")

Largest Landmass Area: 7
Perimeter: 14


In [ ]:
#Task 3
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import math

cities = [
    (0, 0),
    (2, 6),
    (3, 1),
    (5, 5),
    (8, 3),
    (6, 8),
    (1, 7),
    (7, 7),
    (9, 9),
    (4, 9)
]

n = len(cities)
distance_matrix = [[0 for _ in range(n)] for _ in range(n)]
for i in range(n):
    for j in range(n):
        if i != j:
            distance_matrix[i][j] = int(round(math.dist(cities[i], cities[j]) * 100))

print("Variables: x[i][k] indicates whether city i is visited at position k in the tour")
print("Domains: x[i][k] in {0,1}")
print("Constraints: each city visited exactly once, each position filled by one city, return to start city")

manager = pywrapcp.RoutingIndexManager(n, 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_parameters.time_limit.seconds = 5

solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(node + 1)
        next_index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(index, next_index, 0)
        index = next_index
    route.append(manager.IndexToNode(index) + 1)
    print("Optimal Path for 10 Cities:", route)
    print("Total Distance:", route_distance / 100)
else:
    print("No solution found.")

Variables: x[i][k] indicates whether city i is visited at position k in the tour
Domains: x[i][k] in {0,1}
Constraints: each city visited exactly once, each position filled by one city, return to start city
Optimal Path for 10 Cities: [1, 3, 5, 4, 8, 9, 6, 10, 7, 2, 1]
Total Distance: 34.56
